# PA1 Task 2 - DANN stability diagnostics (Kaggle, rerun-safe)

Run this with **Save Version -> Save & Run All (Commit)**. It then executes top to bottom on a fresh
machine and Kaggle keeps everything written to `/kaggle/working` as the version's output, so a dead
interactive session cannot eat your results. Expect roughly 1.5-2 hours for all three runs.

**Why these runs exist.** DANN and CDAN diverge under the handout's configuration at every alignment
strength tested (alpha_max = 1, 0.5, 0.25; the last blew up in epoch 2, when the effective alpha was
0.06). Each variant below removes exactly one suspected cause:

| Run | Change | Cause it tests |
|---|---|---|
| `dann_flip` | backbone minimises CE against **flipped** domain labels (bounded below) instead of maximising CE through the GRL (unbounded above) | the objective has no maximum |
| `dann_l2norm` | the discriminator's input is L2-normalised; GRL and the alpha schedule are untouched | inflating feature scale is the cheapest way up, since frozen BatchNorm never renormalises |
| `dann_sgd` | SGD + momentum + Ganin's annealed learning rate instead of AdamW | Adam's per-parameter normalisation makes alpha and clipping inert |

**What it saves, after every stage, into `/kaggle/working/outputs/`:**
`epochs.csv` (every epoch of every run), `summary.csv` (one row per variant), `sketch_eval.csv`
(Sketch accuracy, macro-F1, separability, per-class), `curves_<run>.png`, and
`diagnostics_full.zip` (runs **including** checkpoints) plus `diagnostics_light.zip` (no checkpoints).

**Inputs to attach** (Add Input -> Your Datasets), then check the slugs in K1:
`pa1-repo`, `pa1-task2-b`, `dann-diagnostics`, `pacs-dataset`.
Optionally attach a previous version's output to resume instead of retraining.

## Part K - setup

In [24]:
# K1. Assemble the repo: base -> task2 code -> diagnostics patch (patch last, so it wins).
import os, glob, shutil, json, math, subprocess, gc
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')                      # figures are saved to disk; also shown inline below
import matplotlib.pyplot as plt

BASE = '/kaggle/input/datasets/zylah628'   # <- change if your datasets sit elsewhere
REPO = '/kaggle/working/PA1'
OUT  = '/kaggle/working/outputs'           # everything worth keeping is written here
RUNS_DIR = 'task2/results/runs_kaggle'     # relative to REPO
os.makedirs(OUT, exist_ok=True)

LAYERS = [f'{BASE}/pa1-repo', f'{BASE}/pa1-task2-b',
          f'{BASE}/dann-diagnostics', f'{BASE}/dann-confusion']

def root_of(d):
    """Kaggle sometimes nests an extracted zip one level deeper."""
    if any(os.path.isdir(f'{d}/{x}') for x in ['task2', 'task1', 'common', 'shared']):
        return d
    subs = [p.rstrip('/') for p in glob.glob(f'{d}/*/') if os.path.isdir(p)]
    return subs[0] if len(subs) == 1 else d

os.makedirs(REPO, exist_ok=True)
for layer in LAYERS:
    assert os.path.isdir(layer), f'Missing dataset: {layer}\nAttached: {os.listdir(BASE)}'
    shutil.copytree(root_of(layer), REPO, dirs_exist_ok=True)
    print('layered in:', os.path.basename(layer))
os.chdir(REPO)

PACS = os.path.dirname(glob.glob(f'{BASE}/pacs-dataset/**/sketch', recursive=True)[0])

# OPTIONAL resume: if a previous version's output is attached as an input, reuse its finished runs
# instead of retraining them.
for prev in glob.glob('/kaggle/input/**/diagnostics_full.zip', recursive=True):
    print('restoring previous runs from', prev)
    os.system(f'unzip -o -q "{prev}" -d "{REPO}"')
    break

NEEDED = ['task2/train.py', 'task2/configs/diag_dann_flip.yaml', 'task2/configs/diag_dann_l2norm.yaml',
          'task2/configs/diag_dann_sgd.yaml', 'shared/engine.py', 'common/io.py',
          'shared/splits/pacs_sources_seed6304.json']
missing = [f for f in NEEDED if not os.path.exists(f)]
print('\nrepo    :', sorted(os.listdir(REPO)))
print('PACS    :', PACS)
print('outputs :', OUT)
print('MISSING :', missing if missing else 'nothing -- ready')
assert not missing

layered in: pa1-repo
layered in: pa1-task2-b
layered in: dann-diagnostics
layered in: dann-confusion

repo    : ['.gitignore', 'README.md', 'common', 'notebooks', 'report', 'requirements.txt', 'shared', 'task1', 'task2']
PACS    : /kaggle/input/datasets/zylah628/pacs-dataset/PACS
outputs : /kaggle/working/outputs
MISSING : nothing -- ready


In [25]:
!grep -c "confusion" task2/methods/dann.py          # expect 2 or more
!ls task2/configs/diag_dann_confusion*.yaml

5
task2/configs/diag_dann_confusion_l2.yaml
task2/configs/diag_dann_confusion.yaml


In [26]:
# K2. Helpers. `report()` prints a health summary, saves the curves as a PNG, and returns the row
# that goes into summary.csv. `save_tables()` rewrites the CSVs from whatever has finished so far, so
# the outputs stay valid even if a later run fails.
CHANCE_LOSS = math.log(7)          # 1.946 = cross-entropy of uniform guessing over 7 classes

def run(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Command failed (exit code {p.returncode}): {cmd}')

def verdict(hist, summary):
    """Stricter than the earlier version, which wrongly passed dann_sgd: a run that peaks and then
    degrades is NOT a clean run."""
    cls = [h['loss_cls'] for h in hist]
    f1 = [h['val_mean_macro_f1'] for h in hist]
    best = summary['best_mean_macro_f1']
    if min(cls) > 1.6 or best < 50:
        return 'COLLAPSED (never learned the classes)'
    degraded = f1[-1] < best - 10 or cls[-1] > max(0.5, 3 * min(cls))
    if degraded:
        return 'DEGRADED (learned, then diverged)'
    return 'STABLE' if best >= 90 else 'WEAK (stable but low source F1)'

def report(run_name, show=True):
    d = f'{RUNS_DIR}/{run_name}'
    s, hist = json.load(open(f'{d}/summary.json')), json.load(open(f'{d}/history.json'))
    ep = [h['epoch'] for h in hist]; f1 = [h['val_mean_macro_f1'] for h in hist]
    cls = [h['loss_cls'] for h in hist]; gn = [h.get('grad_norm', float('nan')) for h in hist]
    fn = [h.get('feat_norm', float('nan')) for h in hist]
    v = verdict(hist, s)
    print(f'\n=== {run_name} ===')
    print(f'  epochs run              : {s["epochs_run"]} of {s["config"]["max_epochs"]}')
    print(f'  best source-val macro-F1: {s["best_mean_macro_f1"]:.2f} (epoch {ep[int(pd.Series(f1).idxmax())]}),'
          f' last epoch {f1[-1]:.2f}')
    print(f'  classification loss     : {cls[0]:.3f} -> {cls[-1]:.3f}  (min {min(cls):.3f}; 1.95 = chance)')
    for k, lab in [('loss_dom', 'domain loss            '), ('loss_gen', 'generator (flip) loss  ')]:
        if k in hist[0]:
            print(f'  {lab} : {hist[0][k]:.3f} -> {hist[-1][k]:.3f}'
                  + ('   (0.69 = ln 2, the equilibrium)' if k == 'loss_dom' else ''))
    if 'disc_acc' in hist[0]:
        print(f'  discriminator accuracy  : {100*hist[0]["disc_acc"]:.1f}% -> {100*hist[-1]["disc_acc"]:.1f}%'
              f'   (50% = chance; BELOW 50% means the features carry INVERTED domain information)')
    if fn[0] == fn[0]:
        print(f'  mean feature norm       : {fn[0]:.2f} -> {fn[-1]:.2f}  (x{fn[-1]/max(fn[0],1e-9):.1f})')
    print(f'  max gradient norm       : {max(gn):.3g}')
    print(f'  VERDICT: {v}')

    panels = [('loss_cls', 'classification loss', CHANCE_LOSS)]
    if 'loss_dom' in hist[0]: panels.append(('loss_dom', 'domain loss', math.log(2)))
    if 'disc_acc' in hist[0]: panels.append(('disc_acc', 'discriminator accuracy', 0.5))
    if fn[0] == fn[0]: panels.append(('feat_norm', 'mean feature norm', None))
    panels += [('val_mean_macro_f1', 'mean source-val macro-F1', None), ('grad_norm', 'gradient norm', None)]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 2.6))
    for ax, (key, title, line) in zip(axes, panels):
        y = [h.get(key, float('nan')) for h in hist]
        ax.plot(ep, y, marker='.')
        if max(y) > 100 * max(min(y), 1e-6): ax.set_yscale('log')
        if line: ax.axhline(line, color='r', ls=':', lw=1)
        ax.set_title(f'{run_name}: {title}', fontsize=8); ax.set_xlabel('epoch'); ax.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(f'{OUT}/curves_{run_name}.png', dpi=140)
    if show:
        plt.show()
    plt.close(fig)

    return {'run': run_name, 'epochs': s['epochs_run'], 'best_epoch': ep[int(pd.Series(f1).idxmax())],
            'best_src_val_f1': round(s['best_mean_macro_f1'], 2), 'last_src_val_f1': round(f1[-1], 2),
            'min_loss_cls': round(min(cls), 3), 'final_loss_cls': round(cls[-1], 3),
            'final_loss_dom': round(hist[-1].get('loss_dom', float('nan')), 3),
            'final_disc_acc': round(hist[-1].get('disc_acc', float('nan')), 3),
            'feat_norm_first': round(fn[0], 2) if fn[0] == fn[0] else None,
            'feat_norm_last': round(fn[-1], 2) if fn[-1] == fn[-1] else None,
            'max_grad_norm': float(f'{max(gn):.4g}'), 'verdict': v}

def save_tables(summary_rows):
    """Rewrite epochs.csv and summary.csv from everything finished so far."""
    ep_rows = []
    for r in sorted(os.listdir(RUNS_DIR)) if os.path.isdir(RUNS_DIR) else []:
        h = f'{RUNS_DIR}/{r}/history.json'
        if os.path.exists(h):
            for e in json.load(open(h)):
                ep_rows.append({'run': r, **e})
    if ep_rows:
        pd.DataFrame(ep_rows).to_csv(f'{OUT}/epochs.csv', index=False)
    if summary_rows:
        pd.DataFrame(summary_rows).to_csv(f'{OUT}/summary.csv', index=False)
    print(f'saved -> {OUT}/epochs.csv ({len(ep_rows)} rows), {OUT}/summary.csv ({len(summary_rows)} rows)')

print('Helpers ready.')

Helpers ready.


In [27]:
# K3. Check the data and splits. The splits shipped with the repo and must match this PACS copy
# image for image, otherwise these runs are not comparable with the Colab ones.
EXPECTED = {'photo': 1670, 'art_painting': 2048, 'cartoon': 2344, 'sketch': 3929}
for d, n in EXPECTED.items():
    got = len([p for p in glob.glob(f'{PACS}/{d}/*/*') if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {d:13s} {got:5d}  (expected {n})' + ('' if got == n else '   <-- MISMATCH, stop here'))
src = json.load(open('shared/splits/pacs_sources_seed6304.json'))['domains']
print('\nSplits:', {d: f"train {len(s['train'])}, val {len(s['val'])}" for d, s in src.items()})
import torch
print('torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- set Accelerator to GPU')

  photo          1670  (expected 1670)
  art_painting   2048  (expected 2048)
  cartoon        2344  (expected 2344)
  sketch         3929  (expected 3929)

Splits: {'photo': 'train 1336, val 334', 'art_painting': 'train 1638, val 410', 'cartoon': 'train 1875, val 469'}
torch 2.10.0+cu128 | GPU: Tesla T4


## Part T - training

**What "fixed" looks like:** `loss_cls` well below 0.5 and staying there, source-validation macro-F1
in the 90s, and no late-run climb in the losses. A run that peaks and then degrades counts as
DEGRADED, not stable.

In [28]:
# K4. Train the three variants. Finished runs are skipped, and the CSVs are rewritten after each
# one, so partial progress is always saved. Roughly 25-40 minutes per run.
COMMON = f'--set data_root={PACS} results_dir={RUNS_DIR}'
DIAGS = [('diag_dann_flip', 'dann_flip'), ('diag_dann_l2norm', 'dann_l2norm'), ('diag_dann_sgd', 'dann_sgd')]

summary_rows = []
for c, r in DIAGS:
    if os.path.exists(f'{RUNS_DIR}/{r}/summary.json'):
        print(f'{r}: already finished -- skipping')
    else:
        run(f'python -m task2.train --config task2/configs/{c}.yaml {COMMON}')
    summary_rows.append(report(r))
    save_tables(summary_rows)

dann_flip: already finished -- skipping

=== dann_flip ===
  epochs run              : 16 of 30
  best source-val macro-F1: 93.22 (epoch 11), last epoch 91.64
  classification loss     : 0.538 -> 0.095  (min 0.095; 1.95 = chance)
  domain loss             : 0.873 -> 0.948   (0.69 = ln 2, the equilibrium)
  generator (flip) loss   : 0.910 -> 0.599
  discriminator accuracy  : 52.5% -> 32.4%   (50% = chance; BELOW 50% means the features carry INVERTED domain information)
  mean feature norm       : 54.42 -> 1193.82  (x21.9)
  max gradient norm       : 43.7
  VERDICT: STABLE
saved -> /kaggle/working/outputs/epochs.csv (36 rows), /kaggle/working/outputs/summary.csv (1 rows)
dann_l2norm: already finished -- skipping

=== dann_l2norm ===
  epochs run              : 13 of 30
  best source-val macro-F1: 93.69 (epoch 8), last epoch 92.51
  classification loss     : 0.549 -> 0.063  (min 0.063; 1.95 = chance)
  domain loss             : 0.703 -> 0.721   (0.69 = ln 2, the equilibrium)
  discriminat

In [29]:
# K4b. The confusion objective: optimum is a discriminator at chance, so inversion no longer pays.
open('task2/configs/diag_cdan_confusion.yaml', 'w').write(
"""base: cdan.yaml
run_name: cdan_confusion
method:
  adv_loss: confusion
""")

for c, r in [('diag_dann_confusion', 'dann_confusion'),
             ('diag_cdan_confusion', 'cdan_confusion')]:
    if not os.path.exists(f'{RUNS_DIR}/{r}/summary.json'):
        run(f'python -m task2.train --config task2/configs/{c}.yaml {COMMON}')
    summary_rows.append(report(r))
    save_tables(summary_rows)

$ python -m task2.train --config task2/configs/diag_dann_confusion.yaml --set data_root=/kaggle/input/datasets/zylah628/pacs-dataset/PACS results_dir=task2/results/runs_kaggle
[dann_confusion] ep01 loss_cls=0.5448 loss_dom=0.4346 loss_gen=0.9346 disc_acc=0.8245 alpha=0.0825 loss_total=0.6210 | val mean F1=88.15
[dann_confusion] ep02 loss_cls=0.2555 loss_dom=0.5052 loss_gen=0.8852 disc_acc=0.7738 alpha=0.2440 loss_total=0.4705 | val mean F1=92.40
[dann_confusion] ep03 loss_cls=0.2002 loss_dom=0.5197 loss_gen=0.8184 disc_acc=0.7593 alpha=0.3930 loss_total=0.5216 | val mean F1=90.28
[dann_confusion] ep04 loss_cls=0.1443 loss_dom=0.5455 loss_gen=0.7934 disc_acc=0.7323 alpha=0.5239 loss_total=0.5600 | val mean F1=92.18
[dann_confusion] ep05 loss_cls=0.1132 loss_dom=0.5701 loss_gen=0.7722 disc_acc=0.7170 alpha=0.6340 loss_total=0.6029 | val mean F1=90.48
[dann_confusion] ep06 loss_cls=0.0915 loss_dom=0.5710 loss_gen=0.7642 disc_acc=0.7234 alpha=0.7233 loss_total=0.6441 | val mean F1=93.73
[d

In [30]:
# K5. The variants side by side, with the original DANN runs from Colab for reference
# (their histories ship with the repo dataset; they predate feat_norm logging, hence the blanks).
ref_rows = []
for r in ['dann', 'dann_alpha0.25', 'dann_alpha0.5', 'source_only']:
    p = f'task2/results/runs_clip0.0/{r}/summary.json'
    if not os.path.exists(p):
        continue
    s, h = json.load(open(p)), json.load(open(f'task2/results/runs_clip0.0/{r}/history.json'))
    cls = [e['loss_cls'] for e in h]; f1 = [e['val_mean_macro_f1'] for e in h]
    ref_rows.append({'run': r + ' (colab)', 'epochs': s['epochs_run'],
                     'best_src_val_f1': round(s['best_mean_macro_f1'], 2), 'last_src_val_f1': round(f1[-1], 2),
                     'min_loss_cls': round(min(cls), 3), 'final_loss_cls': round(cls[-1], 3),
                     'max_grad_norm': float(f"{max(e.get('grad_norm', float('nan')) for e in h):.4g}"),
                     'verdict': verdict(h, s)})
tbl = pd.DataFrame(summary_rows + ref_rows)
display(tbl)
tbl.to_csv(f'{OUT}/summary.csv', index=False)
print('saved ->', f'{OUT}/summary.csv')

,run,epochs,best_epoch,best_src_val_f1,last_src_val_f1,min_loss_cls,final_loss_cls,final_loss_dom,final_disc_acc,feat_norm_first,feat_norm_last,max_grad_norm,verdict
0,dann_flip,16,11.0,93.22,91.64,0.095,0.095,0.948,0.324,54.42,1193.82,43.73,STABLE
1,dann_l2norm,13,8.0,93.69,92.51,0.063,0.063,0.721,0.342,50.54,368.10,13.28,STABLE
2,dann_sgd,7,2.0,86.75,64.85,0.401,1.315,3.126,0.457,30.70,1249.93,300.10,"DEGRADED (learned, then diverged)"
3,dann_confusion,11,6.0,93.73,91.37,0.060,0.070,0.574,0.695,39.13,102.90,14.02,STABLE
4,cdan_confusion,9,4.0,93.87,93.09,0.242,0.242,0.548,0.714,35.19,21.69,11.84,STABLE
5,dann (colab),23,NaN,35.04,4.83,1.489,31.362,NaN,NaN,NaN,NaN,10030000.00,COLLAPSED (never learned the classes)
6,source_only (colab),20,NaN,94.14,91.42,0.025,0.052,NaN,NaN,NaN,NaN,11.17,STABLE


saved -> /kaggle/working/outputs/summary.csv


## Part E - Sketch evaluation

In [31]:
# K6. Sketch evaluation for every variant that has a checkpoint. This is the only use of Sketch
# LABELS here, and it happens after all the runs are fixed.
from task2.evaluate_final import load_model
from shared.diagnostics import extract, separability
from shared.pacs_protocol import load_source_splits, load_target_items
from common.metrics import classification_metrics
from shared.pacs import CLASSES

splits, target = load_source_splits(), load_target_items()
src_val = [it for d in splits for it in splits[d]['val']]
print(f'{len(src_val)} source-validation images, {len(target)} sketches')

rows = []
for _, r in DIAGS:
    d = f'{RUNS_DIR}/{r}'
    if not os.path.exists(f'{d}/best.pt'):
        print(f'{r}: no checkpoint, skipping'); continue
    model, _ = load_model(d, 'cuda')
    fs, _, _ = extract(model, PACS, src_val, 'cuda', batch_size=64)
    ft, zt, yt = extract(model, PACS, target, 'cuda', batch_size=64)
    m = classification_metrics(yt, zt.argmax(1), len(CLASSES))
    rows.append({'run': r, 'sketch_acc': round(m['acc'], 2), 'sketch_f1': round(m['macro_f1'], 2),
                 'separability': round(separability([fs, ft]), 2),
                 **{c: round(v, 1) for c, v in zip(CLASSES, m['per_class_acc'])}})
    print(rows[-1])
    del model, fs, ft, zt; gc.collect(); torch.cuda.empty_cache()

if rows:
    sk = pd.DataFrame(rows)
    display(sk)
    sk.to_csv(f'{OUT}/sketch_eval.csv', index=False)
    print('saved ->', f'{OUT}/sketch_eval.csv')
print('\nReference from the main Colab runs (same splits):')
print('  source_only 69.59% acc, separability 99.86 | dan 58.59 / 82.28 | dann 26.44 / 98.08')
print('\nWatch the separability column: the discriminator ends BELOW chance in the working variants,')
print('so the features may carry inverted domain information that a fresh probe still reads easily.')

1213 source-validation images, 3929 sketches
{'run': 'dann_flip', 'sketch_acc': 5.37, 'sketch_f1': 3.34, 'separability': 100.0, 'dog': 0.0, 'elephant': 0.0, 'giraffe': 0.0, 'guitar': 8.2, 'horse': 0.1, 'house': 0.0, 'person': 100.0}
{'run': 'dann_l2norm', 'sketch_acc': 23.42, 'sketch_f1': 9.16, 'separability': 100.0, 'dog': 0.0, 'elephant': 0.0, 'giraffe': 0.0, 'guitar': 17.1, 'horse': 100.0, 'house': 0.0, 'person': 0.0}
{'run': 'dann_sgd', 'sketch_acc': 55.89, 'sketch_f1': 53.17, 'separability': 100.0, 'dog': 12.0, 'elephant': 93.1, 'giraffe': 60.6, 'guitar': 81.6, 'horse': 36.5, 'house': 98.8, 'person': 53.1}


,run,sketch_acc,sketch_f1,separability,dog,elephant,giraffe,guitar,horse,house,person
0,dann_flip,5.37,3.34,100.0,0.0,0.0,0.0,8.2,0.1,0.0,100.0
1,dann_l2norm,23.42,9.16,100.0,0.0,0.0,0.0,17.1,100.0,0.0,0.0
2,dann_sgd,55.89,53.17,100.0,12.0,93.1,60.6,81.6,36.5,98.8,53.1


saved -> /kaggle/working/outputs/sketch_eval.csv

Reference from the main Colab runs (same splits):
  source_only 69.59% acc, separability 99.86 | dan 58.59 / 82.28 | dann 26.44 / 98.08

Watch the separability column: the discriminator ends BELOW chance in the working variants,
so the features may carry inverted domain information that a fresh probe still reads easily.


In [37]:
# K6b. Sketch evaluation for EVERY run in RUNS_DIR that has a checkpoint.
from task2.evaluate_final import load_model
from shared.diagnostics import extract, separability
from shared.pacs_protocol import load_source_splits, load_target_items
from common.metrics import classification_metrics
from shared.pacs import CLASSES
import gc, torch

splits, target = load_source_splits(), load_target_items()
src_val = [it for d in splits for it in splits[d]['val']]
rows = []
for r in sorted(os.listdir(RUNS_DIR)):
    if not os.path.exists(f'{RUNS_DIR}/{r}/best.pt'):
        continue
    model, _ = load_model(f'{RUNS_DIR}/{r}', 'cuda')
    fs, zs, ys = extract(model, PACS, src_val, 'cuda', batch_size=64)
    ft, zt, yt = extract(model, PACS, target, 'cuda', batch_size=64)
    m = classification_metrics(yt, zt.argmax(1), len(CLASSES))
    rows.append({'run': r, 'src_val_acc': round(classification_metrics(ys, zs.argmax(1), 7)['acc'], 2),
                 'sketch_acc': round(m['acc'], 2), 'sketch_f1': round(m['macro_f1'], 2),
                 'separability': round(separability([fs, ft]), 2),
                 'n_classes_predicted': len(set(zt.argmax(1))),
                 **{c: round(v, 1) for c, v in zip(CLASSES, m['per_class_acc'])}})
    print(rows[-1])
    del model, fs, ft, zs, zt; gc.collect(); torch.cuda.empty_cache()

sk = pd.DataFrame(rows)
display(sk)
sk.to_csv(f'{OUT}/sketch_eval.csv', index=False)
print('\nReference: source_only 69.59% / sep 99.86 | dan 58.59 / 82.28 | dann 26.44 / 98.08')

{'run': 'cdan_confusion', 'src_val_acc': 93.73, 'sketch_acc': 19.55, 'sketch_f1': 5.3, 'separability': 100.0, 'n_classes_predicted': 2, 'dog': 0.0, 'elephant': 0.0, 'giraffe': 100.0, 'guitar': 2.5, 'horse': 0.0, 'house': 0.0, 'person': 0.0}
{'run': 'dann_flip', 'src_val_acc': 92.91, 'sketch_acc': 5.37, 'sketch_f1': 3.34, 'separability': 100.0, 'n_classes_predicted': 3, 'dog': 0.0, 'elephant': 0.0, 'giraffe': 0.0, 'guitar': 8.2, 'horse': 0.1, 'house': 0.0, 'person': 100.0}
{'run': 'dann_l2norm', 'src_val_acc': 92.99, 'sketch_acc': 23.42, 'sketch_f1': 9.16, 'separability': 100.0, 'n_classes_predicted': 2, 'dog': 0.0, 'elephant': 0.0, 'giraffe': 0.0, 'guitar': 17.1, 'horse': 100.0, 'house': 0.0, 'person': 0.0}
{'run': 'dann_sgd', 'src_val_acc': 86.4, 'sketch_acc': 55.89, 'sketch_f1': 53.17, 'separability': 100.0, 'n_classes_predicted': 7, 'dog': 12.0, 'elephant': 93.1, 'giraffe': 60.6, 'guitar': 81.6, 'horse': 36.5, 'house': 98.8, 'person': 53.1}


,run,src_val_acc,sketch_acc,sketch_f1,separability,n_classes_predicted,dog,elephant,giraffe,guitar,horse,house,person
0,cdan_confusion,93.73,19.55,5.30,100.00,2,0.0,0.0,100.0,2.5,0.0,0.0,0.0
1,dann_confusion,93.57,5.09,2.68,99.73,5,0.1,0.0,3.9,1.5,0.1,0.0,100.0
2,dann_flip,92.91,5.37,3.34,100.00,3,0.0,0.0,0.0,8.2,0.1,0.0,100.0
3,dann_l2norm,92.99,23.42,9.16,100.00,2,0.0,0.0,0.0,17.1,100.0,0.0,0.0
4,dann_sgd,86.40,55.89,53.17,100.00,7,12.0,93.1,60.6,81.6,36.5,98.8,53.1



Reference: source_only 69.59% / sep 99.86 | dan 58.59 / 82.28 | dann 26.44 / 98.08


## Part S - save

In [38]:
# K7. Package everything. Run this even if an earlier cell failed.
os.system(f'cd {REPO} && zip -qr {OUT}/diagnostics_full.zip {RUNS_DIR}')              # with checkpoints
os.system(f'cd {REPO} && zip -qr {OUT}/diagnostics_light.zip {RUNS_DIR} -x "*.pt"')   # without
for f in sorted(os.listdir(OUT)):
    print(f'{os.path.getsize(f"{OUT}/{f}")/1e6:8.2f} MB  outputs/{f}')
print('\nAfter a committed run these are under the version\'s Output tab; download diagnostics_full.zip')
print('(it holds the checkpoints) and the CSVs, then copy them into Drive next to your Colab results.')

    0.08 MB  outputs/curves_cdan_confusion.png
    0.10 MB  outputs/curves_dann_confusion.png
    0.10 MB  outputs/curves_dann_flip.png
    0.10 MB  outputs/curves_dann_l2norm.png
    0.09 MB  outputs/curves_dann_sgd.png
  213.01 MB  outputs/diagnostics_full.zip
    0.02 MB  outputs/diagnostics_light.zip
    0.02 MB  outputs/epochs.csv
    0.00 MB  outputs/sketch_eval.csv
    0.00 MB  outputs/summary.csv

After a committed run these are under the version's Output tab; download diagnostics_full.zip
(it holds the checkpoints) and the CSVs, then copy them into Drive next to your Colab results.


In [33]:
open('task2/configs/diag_cdan_confusion.yaml', 'w').write(
"""base: cdan.yaml
run_name: cdan_confusion
method:
  adv_loss: confusion
""")

for c, r in [('diag_dann_confusion', 'dann_confusion'),
             ('diag_cdan_confusion', 'cdan_confusion')]:
    if not os.path.exists(f'{RUNS_DIR}/{r}/summary.json'):
        run(f'python -m task2.train --config task2/configs/{c}.yaml {COMMON}')
    summary_rows.append(report(r))
    save_tables(summary_rows)


=== dann_confusion ===
  epochs run              : 11 of 30
  best source-val macro-F1: 93.73 (epoch 6), last epoch 91.37
  classification loss     : 0.545 -> 0.070  (min 0.060; 1.95 = chance)
  domain loss             : 0.435 -> 0.574   (0.69 = ln 2, the equilibrium)
  generator (flip) loss   : 0.935 -> 0.783
  discriminator accuracy  : 82.4% -> 69.5%   (50% = chance; BELOW 50% means the features carry INVERTED domain information)
  mean feature norm       : 39.13 -> 102.90  (x2.6)
  max gradient norm       : 14
  VERDICT: STABLE
saved -> /kaggle/working/outputs/epochs.csv (56 rows), /kaggle/working/outputs/summary.csv (6 rows)

=== cdan_confusion ===
  epochs run              : 9 of 30
  best source-val macro-F1: 93.87 (epoch 4), last epoch 93.09
  classification loss     : 0.541 -> 0.242  (min 0.242; 1.95 = chance)
  domain loss             : 0.384 -> 0.548   (0.69 = ln 2, the equilibrium)
  generator (flip) loss   : 1.117 -> 0.829
  discriminator accuracy  : 87.0% -> 71.4%   (50% 

In [34]:
# Is the collapse real, or is the checkpoint broken? Check source accuracy with the SAME eval path.
from task2.evaluate_final import load_model
from shared.diagnostics import extract
from shared.pacs_protocol import load_source_splits, load_target_items
from common.metrics import classification_metrics
from shared.pacs import CLASSES
import numpy as np, collections

splits, target = load_source_splits(), load_target_items()
src_val = [it for d in splits for it in splits[d]['val']]
for r in ['dann_flip', 'dann_l2norm']:
    model, _ = load_model(f'{RUNS_DIR}/{r}', 'cuda')
    fs, zs, ys = extract(model, PACS, src_val, 'cuda', batch_size=64)
    ft, zt, yt = extract(model, PACS, target, 'cuda', batch_size=64)
    ms = classification_metrics(ys, zs.argmax(1), 7)
    pt = zt.argmax(1)
    top = collections.Counter(pt).most_common(1)[0]
    print(f'\n{r}')
    print(f'  source-val acc (eval path): {ms["acc"]:.2f}   <- should be ~93-94, matching training')
    print(f'  target: {len(set(pt))} distinct classes predicted; modal class {CLASSES[top[0]]} on {100*top[1]/len(pt):.1f}%')
    print(f'  feature std  source {fs.std():.3f} | target {ft.std():.3f}')
    print(f'  logit range  source {zs.min():.1f}..{zs.max():.1f} | target {zt.min():.1f}..{zt.max():.1f}')


dann_flip
  source-val acc (eval path): 92.91   <- should be ~93-94, matching training
  target: 3 distinct classes predicted; modal class person on 98.7%
  feature std  source 13.170 | target 19.470
  logit range  source -49.5..81.6 | target -49.1..56.7

dann_l2norm
  source-val acc (eval path): 92.99   <- should be ~93-94, matching training
  target: 2 distinct classes predicted; modal class horse on 97.1%
  feature std  source 8.463 | target 24.585
  logit range  source -33.6..52.0 | target -41.4..15.3


In [35]:
# Write the CDAN confusion config, then queue both runs.
open('task2/configs/diag_cdan_confusion.yaml', 'w').write(
"""base: cdan.yaml
run_name: cdan_confusion
method:
  adv_loss: confusion
""")

for c, r in [('diag_dann_confusion', 'dann_confusion'),
             ('diag_cdan_confusion', 'cdan_confusion')]:
    if not os.path.exists(f'{RUNS_DIR}/{r}/summary.json'):
        run(f'python -m task2.train --config task2/configs/{c}.yaml {COMMON}')
    summary_rows.append(report(r))
    save_tables(summary_rows)


=== dann_confusion ===
  epochs run              : 11 of 30
  best source-val macro-F1: 93.73 (epoch 6), last epoch 91.37
  classification loss     : 0.545 -> 0.070  (min 0.060; 1.95 = chance)
  domain loss             : 0.435 -> 0.574   (0.69 = ln 2, the equilibrium)
  generator (flip) loss   : 0.935 -> 0.783
  discriminator accuracy  : 82.4% -> 69.5%   (50% = chance; BELOW 50% means the features carry INVERTED domain information)
  mean feature norm       : 39.13 -> 102.90  (x2.6)
  max gradient norm       : 14
  VERDICT: STABLE
saved -> /kaggle/working/outputs/epochs.csv (56 rows), /kaggle/working/outputs/summary.csv (8 rows)

=== cdan_confusion ===
  epochs run              : 9 of 30
  best source-val macro-F1: 93.87 (epoch 4), last epoch 93.09
  classification loss     : 0.541 -> 0.242  (min 0.242; 1.95 = chance)
  domain loss             : 0.384 -> 0.548   (0.69 = ln 2, the equilibrium)
  generator (flip) loss   : 1.117 -> 0.829
  discriminator accuracy  : 87.0% -> 71.4%   (50% 

In [36]:
!grep -c "confusion" task2/methods/dann.py        # 0 = patch missing
!ls task2/configs/diag_dann_confusion.yaml

5
task2/configs/diag_dann_confusion.yaml


## Notes for the report
- These are **diagnostics**, not the required comparison. The main Task 2 table keeps the handout's DANN and CDAN, which diverge; these runs explain why.
- **Kaggle uses different hardware from Colab**, so these are not bit-comparable with the main runs. That is fine for a "does it train at all" question; keep the six main runs on their original machine.
- **`dann_l2norm` is the smaller deviation** to defend: it keeps the gradient reversal layer and the alpha schedule exactly as specified and only normalises the discriminator's input.
- If you need to retrain later, attach this version's output as an input dataset; K1 will restore the finished runs from `diagnostics_full.zip` and skip them.